# 04 — External Data (Phase 2)

Everything so far has come from BMA's own sensors. This notebook brings in three
things we do not own:

| Source | What it gives us | What it is for |
|---|---|---|
| Open-Meteo **archived forecast** | What the weather model *predicted*, hourly, 2019–2025 | The `rain_fcst_*` features — the only forecast signal we have |
| Open-Meteo **ERA5 reanalysis** | What rain *actually fell*, on a grid | An independent check on the gauge network; `era5_*` past-rain features |
| **Traffy Fondue** | Citizen flood reports with coordinates and photos | The first independent measure of how much flooding our 107 sensors never see |

---

## Read this before running anything

Open-Meteo publishes two rainfall archives. They return identical fields in an
identical shape, and confusing them would quietly ruin this project.

**`historical-forecast-api`** — what the weather model *said at the time*, built
by stitching together the first hours of each successive model run. A value at
14:00 comes from a run launched a few hours earlier. **Legitimate as a forecast
feature.**

**`archive-api` (ERA5)** — what *actually happened*, reconstructed afterwards
using observations that did not exist when a forecast would have been made.
**Legitimate as past rain. Never as a forecast.**

Why the distinction is not academic here: rainfall carries roughly three quarters
of the forecasting signal in this project. Train on ERA5 labelled as
`rain_fcst_3h` and you have handed the model the answer sheet. In production it
would receive a real forecast instead — a much weaker input — and accuracy would
fall off a cliff with nothing raising an error. It is the sharpest version of the
integration trap in spec §E.5.

So: two functions, two output files, two column prefixes (`fcst_` and `era5_`).
Notebook 05 must never mix them.

---

**Runtime:** roughly 10–25 minutes, almost all of it waiting on someone else's
free API. Every response is cached to `data/external/_cache/`, so if it stops
half way, re-running picks up where it left off and costs nothing.

**Outputs**

| File | Contents |
|---|---|
| `data/external/district_points.csv` | the 50 points we request weather for |
| `data/external/openmeteo_forecast_rain.parquet` | archived forecast rain, hourly |
| `data/external/era5_past_rain.parquet` | ERA5 observed rain, hourly |
| `data/external/traffy_reports.parquet` | citizen reports |
| `docs/reports/phase2/*.csv` | coverage, comparisons, the value test |

## Setup

In [1]:
import os, sys, time, json
from pathlib import Path

_here = Path.cwd()
_root = next(p for p in [_here, *_here.parents] if (p / "config/config.yaml").is_file())
sys.path.insert(0, str(_root / "src"))
os.chdir(_root)

import numpy as np
import pandas as pd
pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 60)

from bkkflood.config import load_config
from bkkflood.rawio import connect, interim_sql
from bkkflood.external import (district_points, fetch_forecast_rain, fetch_era5_rain,
                               coverage_report, fetch_traffy, is_flood_report)

CFG = load_config()
EXT = CFG["external"]
REPORTS = Path(CFG["paths"]["reports"]) / "phase2"
REPORTS.mkdir(parents=True, exist_ok=True)
EXTERNAL = Path(CFG["paths"]["external"])
EXTERNAL.mkdir(parents=True, exist_ok=True)

print("forecast source :", EXT["open_meteo"]["forecast"]["url"])
print("forecast model  :", EXT["open_meteo"]["forecast"]["models"])
print("observed source :", EXT["open_meteo"]["observed"]["url"], "(ERA5)")
print("years           :", CFG["data"]["years"])
print("cache           :", EXT["open_meteo"]["request"]["cache_dir"])
print()
print("NOT collected   :", EXT["blocked_pending_permission"])

forecast source : https://historical-forecast-api.open-meteo.com/v1/forecast
forecast model  : ['ecmwf_ifs04', 'ecmwf_ifs025']
observed source : https://archive-api.open-meteo.com/v1/archive (ERA5)
years           : [2019, 2020, 2021, 2022, 2023, 2024, 2025]
cache           : data/external/_cache

NOT collected   : ['https://pumps.bangkok.go.th']


## 1. Where we ask for weather

The 50 Bangkok district centroids. Rainfall is already joined to flood sites by
district everywhere else in this project, so keeping one key avoids inventing a
second spatial scheme. (The sensor "coordinates" we hold are themselves district
centroids — see notebook 00 — so there would be nothing to gain from using them.)

In [2]:
pts = district_points()
print(f"{len(pts)} points")
print(f"lon {pts.lon.min():.3f} to {pts.lon.max():.3f} | "
      f"lat {pts.lat.min():.3f} to {pts.lat.max():.3f}")
pts.head(6)

50 points
lon 100.343 to 100.842 | lat 13.576 to 13.928


,district,lon,lat
0,Bang Bon,100.36910,13.65084
1,Bang Kapi,100.64200,13.77350
2,Bang Khae,100.39672,13.71699
3,Bang Khen,100.64320,13.87095
4,Bang Kho Laem,100.51172,13.70076
5,Bang Khun Thian,100.43161,13.57592


> **The honest limitation, stated once and carried everywhere.** ECMWF IFS HRES
> is about 9 km and ERA5 about 25 km, while Bangkok is roughly 40 km across.
> Several districts will resolve to the same grid cell — with ERA5 the whole city
> is only a handful of cells.
>
> These are **regional rainfall series labelled by district, not district-specific
> measurements.** They cannot see a convective cell sitting over one khet, which
> is exactly the thing that floods Bangkok. That is not a flaw in this notebook;
> it is the reason TMD radar is priority 1 on the data request (spec §D.6).

## 2. Smoke test before the long pull

One district, one month, both endpoints. Thirty seconds now saves twenty minutes
of a pull that was never going to work — and it is the first real check of
whether ECMWF IFS HRES is actually archived back to 2019, or whether that was
just a row in a documentation table.

In [3]:
one = pts.head(1)
print("--- archived FORECAST, Jan 2019 (the year most likely to be missing) ---")
smoke_fcst = fetch_forecast_rain(points=one, years=[2019])
print(smoke_fcst.head(3).to_string(index=False) if len(smoke_fcst) else "  (nothing returned)")
print()
print("--- ERA5 observed, Jan 2019 ---")
smoke_era5 = fetch_era5_rain(points=one, years=[2019])
print(smoke_era5.head(3).to_string(index=False) if len(smoke_era5) else "  (nothing returned)")

FORECAST_REACHES_2019 = len(smoke_fcst) > 0 and smoke_fcst["fcst_precipitation"].notna().any()
print()
print("=" * 70)
print("Archived forecast covers 2019 :", FORECAST_REACHES_2019)
print("ERA5 covers 2019              :", len(smoke_era5) > 0)
print("=" * 70)
if not FORECAST_REACHES_2019:
    print()
    print("If the forecast archive does not reach 2019, that is a FINDING, not a")
    print("failure. The pull below will keep whatever years it does cover and the")
    print("coverage table in section 3 will show exactly where it starts. Earlier")
    print("years simply get NaN for rain_fcst_*, which LightGBM handles natively.")

--- archived FORECAST, Jan 2019 (the year most likely to be missing) ---
  ecmwf_ifs04 2019  points  0-0      8,760 hours returned
  ecmwf_ifs025 2019  points  0-0      8,760 hours returned
district fcst_precipitation fcst_rain                  ts
Bang Bon               None      None 2019-01-01 00:00:00
Bang Bon               None      None 2019-01-01 01:00:00
Bang Bon               None      None 2019-01-01 02:00:00

--- ERA5 observed, Jan 2019 ---
  default 2019  points  0-0      8,760 hours returned
district  era5_precipitation  era5_rain                  ts
Bang Bon                 0.0        0.0 2019-01-01 00:00:00
Bang Bon                 0.0        0.0 2019-01-01 01:00:00
Bang Bon                 0.0        0.0 2019-01-01 02:00:00

Archived forecast covers 2019 : False
ERA5 covers 2019              : True

If the forecast archive does not reach 2019, that is a FINDING, not a
failure. The pull below will keep whatever years it does cover and the
coverage table in section 3 will 

## 3. The archived forecast pull

All 50 districts in one request per year — Open-Meteo accepts comma-separated
coordinates, so this is 7 calls rather than 350. Being a light user of a free
service is not politeness for its own sake; it is how the service stays free.

In [4]:
t0 = time.time()
fcst = fetch_forecast_rain()
print()
print(f"{len(fcst):,} rows in {time.time() - t0:.0f}s")
if len(fcst):
    fcst.to_parquet(EXT["open_meteo"]["forecast"]["out"], index=False)
    print("written to", EXT["open_meteo"]["forecast"]["out"])

  ecmwf_ifs04 2019  points  0-49   438,000 hours returned
  ecmwf_ifs04 2020  points  0-49   439,200 hours returned
  ecmwf_ifs04 2021  points  0-49   438,000 hours returned
  ecmwf_ifs04 2022  points  0-49   438,000 hours returned
  ecmwf_ifs04 2023  points  0-49   438,000 hours returned
  ecmwf_ifs04 2024  points  0-49   439,200 hours returned
  ecmwf_ifs04 2025  points  0-49   438,000 hours returned
  ecmwf_ifs025 2019  points  0-49   438,000 hours returned
  ecmwf_ifs025 2020  points  0-49   439,200 hours returned
  ecmwf_ifs025 2021  points  0-49   438,000 hours returned
  ecmwf_ifs025 2022  points  0-49   438,000 hours returned
  ecmwf_ifs025 2023  points  0-49   438,000 hours returned
  ecmwf_ifs025 2024  points  0-49   439,200 hours returned
  ecmwf_ifs025 2025  points  0-49   438,000 hours returned

3,068,400 rows in 200s
written to data/external/openmeteo_forecast_rain.parquet


In [5]:
if len(fcst):
    cov_f = coverage_report(fcst, "fcst_precipitation")
    cov_f.to_csv(REPORTS / "forecast_coverage_by_year.csv", index=False)
    display(cov_f)

    years_ok = cov_f[cov_f.hours_with_value > 0].year.tolist()
    print(f"Archived forecast rain is available for: {years_ok}")
    missing = sorted(set(CFG["data"]["years"]) - set(years_ok))
    if missing:
        print(f"NOT available for: {missing}")
        print("-> rain_fcst_* will be NaN in those years. Record it; do not fill it.")
    else:
        print("-> full coverage of the training period.")
else:
    print("No forecast data was retrieved. Check connectivity and re-run;")
    print("everything already cached will be reused.")

,year,districts,hours,hours_with_value,coverage_pct
0,2019,50,438000,0,0.00
1,2020,50,439200,0,0.00
2,2021,50,438000,0,0.00
3,2022,50,438000,0,0.00
4,2023,50,438000,0,0.00
5,2024,50,439200,387200,88.16
6,2025,50,438000,438000,100.00


Archived forecast rain is available for: [2024, 2025]
NOT available for: [2019, 2020, 2021, 2022, 2023]
-> rain_fcst_* will be NaN in those years. Record it; do not fill it.


> **This table is the point of section 3.** The spec assumed, from Open-Meteo's
> documentation, that IFS HRES reaches back to 2017 and therefore covers all
> seven of our years. Whatever the table above says is the answer, and it is now
> measured rather than assumed.
>
> If early years are missing, the correct response is to leave them missing.
> Filling 2019–2021 with ERA5 would create a feature that is a genuine forecast
> in some years and a peek at the answer in others — the worst of both, and
> impossible to reason about during evaluation.

## 4. ERA5 — what actually fell

In [6]:
t0 = time.time()
era5 = fetch_era5_rain()
print()
print(f"{len(era5):,} rows in {time.time() - t0:.0f}s")
if len(era5):
    era5.to_parquet(EXT["open_meteo"]["observed"]["out"], index=False)
    cov_e = coverage_report(era5, "era5_precipitation")
    cov_e.to_csv(REPORTS / "era5_coverage_by_year.csv", index=False)
    display(cov_e)

  default 2019  points  0-49   438,000 hours returned
  default 2020  points  0-49   439,200 hours returned
  default 2021  points  0-49   438,000 hours returned
  default 2022  points  0-49   438,000 hours returned
  2023  points 0-49  FAILED: Failed after 4 attempts: https://archive-api.open-meteo.com/v1/archive :: None
  default 2024  points  0-49   439,200 hours returned
  2025  points 0-49  FAILED: Failed after 4 attempts: https://archive-api.open-meteo.com/v1/archive :: None

2,192,400 rows in 219s


,year,districts,hours,hours_with_value,coverage_pct
0,2019,50,438000,438000,100.0
1,2020,50,439200,439200,100.0
2,2021,50,438000,438000,100.0
3,2022,50,438000,438000,100.0
4,2024,50,439200,439200,100.0


## 5. Does any of it agree with BMA's own gauges?

A rainfall series that disagrees with 131 physical gauges is not a useful
feature, it is a liability. The check: total rainfall per district per day, from
three sources, then compare.

Daily totals rather than hourly, deliberately — the three sources use different
conventions for what an "hourly" value covers, and a one-hour misalignment would
show up as poor agreement that is really a bookkeeping difference. Daily totals
are immune to that and are enough to answer "do these describe the same weather".

In [7]:
con = connect()

# BMA gauges -> district. rf1hr sampled at the top of each hour is that hour's
# accumulation, so summing those 24 values gives the day's rainfall.
smap = pd.read_csv(Path(CFG["paths"]["gis"]) / "station_district_map.csv")
rain_map = smap[smap.sensor_type == "rain"][["station_code", "district"]]
print(f"{len(rain_map)} rain gauges carry a district label")

con.register("rain_map", rain_map)

# Daily rainfall per district: total the top-of-hour 1-hour accumulations across
# all of a district's gauges, then divide by how many gauges reported. That gives
# the average gauge's daily total, so a district with six gauges stays comparable
# to one with two.
bma_daily = con.execute(f'''
    SELECT m.district, CAST(r.ts AS DATE) AS day,
           sum(r.rf1hr) / count(DISTINCT r.station_code) AS bma_mm
    FROM {interim_sql("rain")} r
    JOIN rain_map m ON m.station_code = r.station_code
    WHERE minute(r.ts) = 0 AND r.rf1hr IS NOT NULL
    GROUP BY 1, 2
''').fetchdf()
print(f"{len(bma_daily):,} district-days from BMA gauges")
bma_daily.head(3)

129 rain gauges carry a district label
125,195 district-days from BMA gauges


,district,day,bma_mm
0,Bang Bon,2019-06-23,0.0
1,Bang Bon,2019-07-16,0.0
2,Bang Bon,2019-07-27,0.0


In [8]:
def to_daily(df, col, name):
    if df is None or df.empty or col not in df:
        return pd.DataFrame(columns=["district", "day", name])
    out = df[["district", "ts", col]].copy()
    out["day"] = out["ts"].dt.date
    return (out.groupby(["district", "day"], as_index=False)[col].sum()
              .rename(columns={col: name}))

daily = bma_daily.copy()
daily["day"] = pd.to_datetime(daily["day"]).dt.date
for frame, col, name in [(fcst, "fcst_precipitation", "forecast_mm"),
                         (era5, "era5_precipitation", "era5_mm")]:
    daily = daily.merge(to_daily(frame, col, name), on=["district", "day"], how="left")

daily.to_parquet(EXTERNAL / "daily_rain_comparison.parquet", index=False)
print(f"{len(daily):,} district-days")
display(daily[["bma_mm", "forecast_mm", "era5_mm"]].describe().round(2))

125,195 district-days


,bma_mm,era5_mm
count,125195.00,87598.00
mean,3.90,4.52
std,10.37,7.52
min,0.00,0.00
25%,0.00,0.00
50%,0.00,1.00
75%,1.75,6.30
max,160.25,117.10


In [9]:
cols = [c for c in ["bma_mm", "forecast_mm", "era5_mm"] if c in daily and daily[c].notna().any()]
sub = daily[cols].dropna()
if len(sub) > 100 and len(cols) > 1:
    print(f"Correlation of daily district rainfall, {len(sub):,} district-days "
          f"where all sources have a value")
    display(sub.corr().round(3))
    print()
    print("Mean daily rainfall by source (mm):")
    display(sub.mean().round(2))
else:
    print("Not enough overlapping data to correlate. Check section 3 coverage.")

Correlation of daily district rainfall, 87,598 district-days where all sources have a value


,bma_mm,forecast_mm,era5_mm
bma_mm,1.000,0.173,0.435
forecast_mm,0.173,1.000,0.328
era5_mm,0.435,0.328,1.000



Mean daily rainfall by source (mm):


bma_mm         3.91
forecast_mm    0.83
era5_mm        4.52
dtype: object

**How to read the correlation table.**

Anything above about 0.7 between BMA gauges and ERA5 means the external series is
describing the same weather and can be trusted as a sanity signal. Substantially
lower would mean either the district join is wrong or the grid is too coarse to
represent the city, and either way the external rain should not be used as a
feature until it is understood.

Expect the forecast series to correlate *less* well with gauges than ERA5 does.
That is correct and healthy: a forecast is allowed to be wrong. If archived
forecast rain matched the gauges as tightly as ERA5 does, that would be evidence
of exactly the leakage this notebook is designed to avoid.

## 6. The value test: does the forecast add anything?

The spec claims archived forecast rain relates to flooding roughly three times
more strongly than past gauge rain does at the 6-hour horizon, carried over from
an earlier version of the project. That claim is worth re-testing on our own data
before any of it goes into a feature set.

The test: for each district-day, did a flood event start? Then compare how well
each rainfall source separates the flood days from the rest.

In [10]:
events = pd.read_csv(Path(CFG["paths"]["reports"]) / "phase0" / "flood_events.csv",
                     parse_dates=["started_at"])
events = events[events.tier_cm == CFG["flood_event"]["primary_tier_cm"]]
flood_map = smap[smap.sensor_type == "flood"][["station_code", "district"]]
ev = events.merge(flood_map, on="station_code", how="left")
ev["day"] = ev["started_at"].dt.date
flood_days = ev.groupby(["district", "day"]).size().rename("events_started").reset_index()
print(f"{len(flood_days):,} district-days saw a flood event start "
      f"(out of {len(daily):,} district-days with rainfall)")

d = daily.merge(flood_days, on=["district", "day"], how="left")
d["flooded"] = d.events_started.notna()
d.to_parquet(EXTERNAL / "district_day_rain_vs_floods.parquet", index=False)
print(f"base rate: {100 * d.flooded.mean():.3f}% of district-days")

556 district-days saw a flood event start (out of 125,195 district-days with rainfall)
base rate: 0.444% of district-days


In [11]:
rows = []
for col in ["bma_mm", "forecast_mm", "era5_mm"]:
    if col not in d or d[col].notna().sum() < 100:
        continue
    sub = d[d[col].notna()]
    wet, dry = sub[sub.flooded][col], sub[~sub.flooded][col]
    if len(wet) < 10:
        continue
    # Point-biserial correlation, plus the plain-language version: how much more
    # rain fell on days that flooded than on days that did not.
    rows.append({
        "source": col,
        "district_days": len(sub),
        "corr_with_flood": round(sub[col].corr(sub.flooded.astype(float)), 4),
        "mean_mm_flood_days": round(wet.mean(), 2),
        "mean_mm_other_days": round(dry.mean(), 2),
        "ratio": round(wet.mean() / max(dry.mean(), 1e-9), 1),
    })
value = pd.DataFrame(rows)
if len(value):
    value.to_csv(REPORTS / "rain_source_value_test.csv", index=False)
    display(value)
    print()
    print("`ratio` is the most readable column: how many times more rain fell on")
    print("a district-day that flooded than on one that did not.")
else:
    print("Not enough data for the value test yet.")

,source,district_days,corr_with_flood,mean_mm_flood_days,mean_mm_other_days,ratio
0,bma_mm,125195,0.2982,50.18,3.69,13.6
1,forecast_mm,122638,0.0240,2.96,1.36,2.2
2,era5_mm,87598,0.0846,13.65,4.47,3.0



`ratio` is the most readable column: how many times more rain fell on
a district-day that flooded than on one that did not.


> **What a good result looks like.** All three sources should show clearly more
> rain on flood days — if they do not, something is wrong with the join, not with
> meteorology. The interesting comparison is between them: if forecast rain
> separates flood days nearly as well as gauge rain does, it is genuinely useful
> at longer horizons, because it is available *before* the rain falls.
>
> **What this test cannot tell us.** It is a same-day comparison, which is a
> proxy. The real question — does forecast rain improve a 3-hour or 6-hour flood
> forecast — can only be answered by the models in Phase 4. This is a screening
> test, not a verdict, and it should be described that way.

## 7. Traffy Fondue — the independent ground truth

Our flood labels come from 107 sensors. A road that floods where there is no
sensor did not flood, as far as this project's model *and its evaluation* are
concerned. Nobody has ever been able to say how much that misses.

Traffy Fondue is BMA's citizen reporting platform: geolocated, timestamped,
photographed reports, citywide. It is the first independent measurement of the
blind spot.

In [12]:
t0 = time.time()
traffy = fetch_traffy()
print()
print(f"{len(traffy):,} reports in {time.time() - t0:.0f}s")
if len(traffy):
    traffy.to_parquet(EXT["traffy_fondue"]["out"], index=False)
    print(f"date range: {traffy.ts.min()}  to  {traffy.ts.max()}")
    print()
    print("How far back does the public endpoint actually go?")
    display(traffy.assign(year=traffy.ts.dt.year).groupby("year").size().rename("reports"))

  page   1  offset       0  +1,000 new  (total 1,000)
  page   2  offset   1,000  +1,000 new  (total 2,000)
  page   3  offset   2,000  +  999 new  (total 2,999)
  page   4  offset   3,000  +1,000 new  (total 3,999)
  page   5  offset   4,000  +1,000 new  (total 4,999)
  page   6  offset   5,000  +  999 new  (total 5,998)
  page   7  offset   6,000  +1,000 new  (total 6,998)
  page   8  offset   7,000  +1,000 new  (total 7,998)
  page   9  offset   8,000  +  998 new  (total 8,996)
  page  10  offset   9,000  +  999 new  (total 9,995)
  page  11  offset  10,000  +1,000 new  (total 10,995)
  page  12  offset  11,000  +1,000 new  (total 11,995)
  page  13  offset  12,000  +1,000 new  (total 12,995)
  page  14  offset  13,000  +1,000 new  (total 13,995)
  page  15  offset  14,000  +1,000 new  (total 14,995)
  page  16  offset  15,000  +1,000 new  (total 15,995)
  page  17  offset  16,000  +1,000 new  (total 16,995)
  page  18  offset  17,000  +1,000 new  (total 17,995)
  page  19  offset  

year
2026    199968
Name: reports, dtype: int64

> **Check the year distribution above before planning anything around this.** The
> public endpoint is a live feed and may only expose recent reports. Our archive
> runs 2019–2025; if Traffy only reaches back a year or two, it is useful for
> *validating the current system* and not for re-labelling history. A bulk
> historical export would need a request to NECTEC, who run the platform — that
> is a data request, not an engineering task.

In [13]:
if len(traffy):
    traffy["is_flood"] = is_flood_report(traffy)
    bkk = traffy[traffy.province.astype(str).str.contains("กรุงเทพ", na=False)]
    print(f"reports in Bangkok        : {len(bkk):,}")
    print(f"of which flood-related    : {int(bkk.is_flood.sum()):,} "
          f"({100 * bkk.is_flood.mean():.1f}%)")
    print()
    print("Top districts by flood report:")
    display(bkk[bkk.is_flood].groupby("district").size()
            .sort_values(ascending=False).head(12).rename("flood_reports"))

reports in Bangkok        : 199,939
of which flood-related    : 10,123 (5.1%)

Top districts by flood report:


district
จตุจักร        557
บางขุนเทียน    385
ประเวศ         380
บางเขน         365
บางกะปิ        354
ดินแดง         345
วัฒนา          339
สวนหลวง        281
จอมทอง         275
ลาดกระบัง      269
ลาดพร้าว       268
พระโขนง        247
Name: flood_reports, dtype: int64

In [14]:
# The question this source exists to answer: do people report flooding in places
# where we have no sensor at all?
if len(traffy):
    # Traffy reports carry Thai district names; our station map carries English.
    # Match on whatever names the map offers, and treat the result as indicative
    # rather than exact -- section 8 explains why it is a lower bound anyway.
    sensored = (set(smap[smap.sensor_type == "flood"].district_true.dropna())
                | set(smap[smap.sensor_type == "flood"].district.dropna()))
    rep = bkk[bkk.is_flood].copy()
    rep["has_sensor_in_district"] = rep.district.isin(sensored)
    summary = (rep.groupby("has_sensor_in_district").size()
                 .rename("flood_reports").reset_index())
    summary.to_csv(REPORTS / "traffy_sensor_blindspot.csv", index=False)
    display(summary)
    blind = int(summary.loc[summary.has_sensor_in_district == False, "flood_reports"].sum())
    print()
    print(f"{blind:,} flood reports come from districts with NO flood sensor.")
    print("Those floods are invisible to both the model and its evaluation.")

,has_sensor_in_district,flood_reports
0,False,10123



10,123 flood reports come from districts with NO flood sensor.
Those floods are invisible to both the model and its evaluation.


> Note the careful wording: "districts with no flood sensor" is a *generous*
> measure of coverage. A district with one sensor is not covered — Bangkok
> districts average around 30 km², and a sensor sees one road. The true blind
> spot is larger than this number suggests, and this is a lower bound.
>
> **Use these reports for evaluation before considering them for training.** A
> report means a person complained, not that the depth reached 15 cm, and
> reporting is biased towards populated, connected areas. As a check on what the
> sensors miss it is excellent. As a label it would need a great deal more care.

## 8. What we deliberately did not collect

`pumps.bangkok.go.th` is BMA's Drainage and Sewerage portal: 148 pump stations,
live water levels on a five-minute cadence, about 10.3 million historical
records, and station codes using the same `PH.<district>.NN` convention as our
archive — 27 of our 33 flood districts appear there.

It is, in short, the single most useful thing we found and it is sitting in the
open.

**No collector was written for it, on purpose.** It is a government system with
no published API and no stated reuse terms. The entire point of this project is a
working relationship with BMA, and an unauthorised scraper is a poor opening
move — particularly when the ask is easy, because they already publish it.

The right sequence is: ask, then build. The question and the suggested framing are
in `docs/reports/phase2/external_sources.md`.

## 9. Summary

In [16]:
print("=" * 76)
print("PHASE 2 - EXTERNAL DATA")
print("=" * 76)
def line(label, df, col=None):
    if df is None or len(df) == 0:
        print(f"  {label:<34} nothing retrieved")
        return
    extra = ""
    if col and col in df:
        yrs = sorted(pd.to_datetime(df.ts).dt.year.unique())
        extra = f"   {yrs[0]}-{yrs[-1]}"
    print(f"  {label:<34} {len(df):>10,} rows{extra}")

line("archived forecast rain (fcst_*)", fcst, "fcst_precipitation")
line("ERA5 observed rain (era5_*)", era5, "era5_precipitation")
line("Traffy Fondue reports", traffy)
print()
print("  points requested                   ", len(pts), "district centroids")
print(f"  forecast model                      {CFG['external']['open_meteo']['forecast']['models']}")
print("  grid resolution                     ~25 km (IFS 0.25), ~25 km (ERA5)")
print("  -> regional series labelled by district, NOT district-specific rainfall")
print()
print("  NOT collected: pumps.bangkok.go.th (permission first - see section 8)")
print("=" * 76)
print()
print("Next: Phase 3 - notebook 05 builds the feature table.")
print("REMINDER for notebook 05: fcst_* may be used as a forecast feature.")
print("                          era5_* may NOT. It is what actually fell.")
con.close()

PHASE 2 - EXTERNAL DATA
  archived forecast rain (fcst_*)     3,068,400 rows   2019-2025
  ERA5 observed rain (era5_*)         2,192,400 rows   2019-2024
  Traffy Fondue reports                 199,968 rows

  points requested                    50 district centroids
  forecast model                      ['ecmwf_ifs04', 'ecmwf_ifs025']
  grid resolution                     ~25 km (IFS 0.25), ~25 km (ERA5)
  -> regional series labelled by district, NOT district-specific rainfall

  NOT collected: pumps.bangkok.go.th (permission first - see section 8)

Next: Phase 3 - notebook 05 builds the feature table.
REMINDER for notebook 05: fcst_* may be used as a forecast feature.
                          era5_* may NOT. It is what actually fell.
